# 09 · How we build a Temple Bot (A1)

**Reproduces**: [lius.cc/llm/demos/temple-bot](https://lius.cc/llm/demos/temple-bot)

**Idea**: a RAG-only chat bot for temple websites or LINE OA. Drop-in answers for deity functions, ritual procedures, and temple history — no fine-tuning, no extra training data.

**Red lines (built-in client-side regex guard)**: do NOT answer personalized divination questions (拜得到 / 該不該求 / 解籤 / 流年 etc.).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lius-cc/lius-cookbook/blob/main/notebooks/how-we-build-temple-bot.ipynb)

📜 CC0 1.0 · v0.1 preprint (not peer-reviewed)

## 1 · Red-line detection (client-side, no LLM call needed)

Cheap, deterministic guard — runs before hitting the RAG API.

In [ ]:
import re

REDLINE_PATTERNS = [
    r"拜得到", r"有沒有效", r"會不會應", r"該不該", r"可以求", r"能求嗎",
    r"準不準", r"靈不靈", r"運勢", r"命運", r"命格", r"姻緣何時", r"幾時會",
    r"流年", r"八字", r"紫微", r"占卜", r"解籤", r"籤詩", r"問事", r"問神",
]
RX = [re.compile(p) for p in REDLINE_PATTERNS]

def is_redline(q: str) -> bool:
    return any(rx.search(q) for rx in RX)

for test in ["媽祖的職司是什麼", "我拜得到嗎", "三朝醮流程", "姻緣何時會出現"]:
    print(f"  {'❌ REDLINE' if is_redline(test) else '✅ OK     '}  {test}")

## 2 · RAG-as-answer (no LLM generation)

For temple-bot use case, we **don't** generate text — we just return top-N relevant entries from lius.cc and let the user click through. This avoids hallucinations entirely.

In [ ]:
import requests

def temple_bot(question: str):
    if is_redline(question):
        return {
            "kind": "redline",
            "reply": "涉及個人占卜 / 解籤 / 運勢，請洽合格道長或宮廟人員。"
        }
    r = requests.post(
        "https://lius.cc/api/llm-rag",
        json={"q": question, "n": 4},
        timeout=10,
    ).json()
    return {
        "kind": "rag",
        "hits": [
            {"name": h["name"], "type": h["type"], "url": h["url"],
             "snippet": (h.get("summary") or h.get("content") or "")[:160] + "…"}
            for h in r.get("hits", [])
        ]
    }

from pprint import pprint
pprint(temple_bot("媽祖的職司是什麼"))

## 3 · LINE OA webhook (FastAPI)

Drop-in webhook handler — receives LINE message, replies with top-3 RAG hits.

In [ ]:
# pip install fastapi uvicorn requests
# from fastapi import FastAPI, Request
# app = FastAPI()

# @app.post("/line/webhook")
# async def webhook(request: Request):
#     body = await request.json()
#     msg = body["events"][0]["message"]["text"]
#     res = temple_bot(msg)
#     if res["kind"] == "redline":
#         reply = res["reply"]
#     else:
#         reply = "\n\n".join(
#             f"📖 {h['name']}\n{h['snippet']}\n{h['url']}" for h in res["hits"][:3]
#         )
#     # send to LINE Messaging API ...
#     return {"replies": [{"type": "text", "text": reply}]}

## Reproducibility

- **Snapshot**: 2026-05-17
- **RAG API**: lius.cc/api/llm-rag (30 req/min, free)
- **Red-line patterns**: 22 regex rules (see cell 1)
- **No LLM generation** — RAG hits are returned verbatim with links back to lius.cc nodes